## cấu trúc
```
IPipelineStep
      ↓
ValidateNameStep
ValidatePercentStep
ValidateTimeStep
      ↓
PipelineRunner
```

## flow
```
payload
  ↓
pipeline runner
  ↓
validate step 1
  ↓
validate step 2
  ↓
validate step n
```

In [ ]:
using System;
using System.Collections.Generic;
using System.Threading.Tasks;

// ======================================
// DOMAIN PAYLOAD
// ======================================

public class SkyTransferPayload
{
    public string Name { get; set; }

    public double Percent { get; set; }

    public DateTime StartTime { get; set; }

    public DateTime? EndTime { get; set; }
}


// ======================================
// PIPELINE CONTEXT
// ======================================

public class ValidationContext
{
    public SkyTransferPayload Payload { get; set; }
}


// ======================================
// PIPELINE STEP CONTRACT
// ======================================

public interface IPipelineStep<T>
{
    Task Execute(T context);
}


// ======================================
// VALIDATION STEPS
// ======================================

public class ValidateNameStep : IPipelineStep<ValidationContext>
{
    public Task Execute(ValidationContext context)
    {
        if (string.IsNullOrWhiteSpace(context.Payload.Name))
            throw new Exception("name is required");

        return Task.CompletedTask;
    }
}


public class ValidatePercentStep : IPipelineStep<ValidationContext>
{
    public Task Execute(ValidationContext context)
    {
        var percent = context.Payload.Percent;

        if (percent <= 0 || percent > 100)
            throw new Exception("percent must be between 1 and 100");

        return Task.CompletedTask;
    }
}


public class ValidateTimeStep : IPipelineStep<ValidationContext>
{
    public Task Execute(ValidationContext context)
    {
        var start = context.Payload.StartTime;
        var end = context.Payload.EndTime;

        if (end.HasValue && end.Value < start)
            throw new Exception("end time must be after start time");

        return Task.CompletedTask;
    }
}


// ======================================
// PIPELINE RUNNER
// ======================================

public class ValidationPipelineRunner<T>
{
    private readonly IList<IPipelineStep<T>> steps;

    public ValidationPipelineRunner(IList<IPipelineStep<T>> steps)
    {
        this.steps = steps;
    }

    public async Task Run(T context)
    {
        foreach (var step in steps)
        {
            await step.Execute(context);
        }
    }
}


// ======================================
// DEMO
// ======================================

var payload = new SkyTransferPayload
{
    Name = "discount-10",
    Percent = 10,
    StartTime = DateTime.Now,
    EndTime = DateTime.Now.AddDays(10)
};


var pipeline = new ValidationPipelineRunner<ValidationContext>(
    new List<IPipelineStep<ValidationContext>>
    {
        new ValidateNameStep(),
        new ValidatePercentStep(),
        new ValidateTimeStep()
    }
);


await pipeline.Run(new ValidationContext
{
    Payload = payload
});


Console.WriteLine("validation passed");

validation passed
